# Before 답변 검토

**읽는 순서:** 전체 시간 요약 → 질문별 답변 → 인용·검색 근거 → 검토 메모.

2026-10-18의 **재구성한 Before** 실행 `20261018T073552164357Z`를 읽습니다. 저장된 출력이 있어 실행하지 않아도 결과를 볼 수 있습니다. 다시 실행해도 모델 호출·GPU 사용·원시 결과 수정은 없습니다. 재실행에는 비공개 결과 캐시가 필요합니다.

- 전체 시간: 질문 입력부터 응답 스트림 종료까지. 첫 토큰 시간: 첫 답변 조각이 나오기까지.
- 전처리 시간: 질문 이해·검색 등을 포함한 생성 전 처리 시간입니다. 검색만의 시간은 아닙니다.
- `ok`는 응답 수집 성공이며, 답변의 정확성을 의미하지 않습니다. 토큰 사용량과 비용은 미측정입니다.
- 질문당 1회, 순차 실행 14개입니다. 첫 질문의 로딩 영향도 포함하며 운영 성능이나 통계적 우위를 뜻하지 않습니다.
- 이 노트북은 Before를 공개한 열람용입니다. 모델명을 숨긴 정식 블라인드 채점표는 아닙니다. 사람 점수는 자동으로 채우지 않습니다.

원본·실행 조건·백업 해시는 [03 실행 노트북](03_metadata_and_ingest.ipynb), 평가 기준은 [평가 규약](evaluation_protocol.md)을 참고하세요. 정답·holdout 자료와 코퍼스 본문 전체는 이 노트북에 복제하지 않습니다.


## 확정한 Before 정성 평가 — trust-usability-v1

**평가명: 공인중개사 자문을 반영한 Before 정성 평가**

- 목적: 이 답변을 실제 사용자에게 제공했을 때 얼마나 믿고 활용할 수 있는가를 1~5점으로 판단합니다.
- 기준 고정·평가 확정: 사용자, 2026-10-19 대화 확인. 사용자는 공인중개사 자문을 받아 중대한 오류와 평가 확정 여부를 작성했다고 확인했습니다. 자문자 성명·자문 일자·서명이나 별도 공인 인증은 이 기록에서 추정하지 않습니다.
- 조건 반영·법률 정확성·근거 적합성·실무 활용성·이해하기 쉬움을 종합적으로 살핍니다. 항목별 점수의 가중 합산은 아닙니다. [고정된 점수 구간과 비교 규칙](evaluation_protocol.md#확정-종합-정성-척도-trust-usability-v1)을 따릅니다.
- 기존 14개 점수·검토 의견을 확정된 Before 정성 기준선으로 유지합니다. 중대한 오류에 대한 기존 의견은 점수와 별도로 읽으며, 낮은 점수나 ‘우선 수정’ 표시만으로 오류 유무를 새로 추정하지 않습니다.
- Q01/Q09/Q12의 미확정 조건·판본과 Q10의 개별 공고 사실에 대한 보류는 유지합니다. 점수 확정은 해당 사실이나 검색 지표까지 모두 확정했다는 뜻이 아닙니다. Q10은 기존 메모에 명시된 한계 고지·질문 구체화·행동 안내 범위로 평가합니다.
- After에서도 동일 질문·척도·평가 범위와 보류 처리 기준으로 비교합니다. 현재 Before는 모델명을 알고 평가한 기록입니다. 후속 블라인드 비교를 수행한다면 두 답변에 같은 절차를 적용하고 이번 평가와 구분합니다.

이번 변경은 평가 지위와 기준의 기록입니다. 원시 답변·시간·점수 수치를 수정하거나 법률 내용을 새로 검증하지 않았습니다.


In [1]:
import json
import os
import sys
from pathlib import Path
from html import escape
from urllib.parse import urlsplit

import pandas as pd
from IPython.display import HTML, Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from harness.evaluation import percentile

CACHE = Path(os.environ.get("AIGO_CACHE_DIR", Path.home() / ".cache/aigo-ai"))
manifest = json.loads((ROOT / "notebooks/redesign/data/before_manifest.json").read_text(encoding="utf-8"))
RUN_ID = "20261018T073552164357Z"
RESULTS = CACHE / "before" / manifest["corpus_sha256"] / "runs" / RUN_ID
assert RESULTS.exists(), "03에 기록된 비공개 Before 백업을 캐시 위치에 복원해주세요."
items = json.loads((ROOT / "notebooks/redesign/eval/eval_set.json").read_text(encoding="utf-8"))["items"]
answers = {item["id"]: json.loads((RESULTS / f"{item['id']}.json").read_text(encoding="utf-8")) for item in items}

# 질문을 편집한 뒤 옛 응답을 새 질문의 결과처럼 읽는 일을 방지합니다.
assert len(answers) == 14
assert all(answers[q["id"]]["id"] == q["id"] and answers[q["id"]]["question"] == q["question"] for q in items)
assert all(r["run_kind"] == "reconstructed_before" for r in answers.values())
print(f"저장된 Before 응답 {len(answers)}개를 읽었습니다. 모델 호출 없음.")


저장된 Before 응답 14개를 읽었습니다. 모델 호출 없음.


## 전체 시간 요약
빈 시간은 미측정입니다. 아래 표는 원시 밀리초 값을 초로 바꾸어 소수 둘째 자리까지 표시합니다.

In [2]:
summary = pd.DataFrame([
    {"질문 ID": r["id"], "질문": r["question"], "수집 상태": r["status"],
     "전체 시간(초)": r["total_ms"] / 1000 if r["total_ms"] is not None else None,
     "첫 토큰(초)": r["ttft_ms"] / 1000 if r["ttft_ms"] is not None else None,
     "전처리(초)": r["trace"]["preformat_ms"] / 1000 if r["trace"]["preformat_ms"] is not None else None,
     "검색 근거 수": len(r["trace"].get("retrieved_docs") or []),
     "인용 수": len(r["citations"]), "오류": r["error"] or "없음"}
    for r in answers.values()
])
display(summary.style.format({"전체 시간(초)": "{:.2f}", "첫 토큰(초)": "{:.2f}", "전처리(초)": "{:.2f}"}, na_rep="미측정"))
elapsed = summary["전체 시간(초)"].dropna().tolist()
print(f"전체 시간: P50 {percentile(elapsed, 0.5):.2f}초 / P95 {percentile(elapsed, 0.95):.2f}초 (n={len(elapsed)})")


,질문 ID,질문,수집 상태,전체 시간(초),첫 토큰(초),전처리(초),검색 근거 수,인용 수,오류
0,Q01,월세 계약은 1년을 했는데 중간에 나가야할 일이 생겼는데 어떻게 해야할까?,ok,24.29,7.54,5.53,5,5,없음
1,Q02,"내가 지금 전세금 3억정도의 집에 살고있는데, 전세 계약 만료일이 1년 정도 남았단 말이야? 근데 특약으로 전세권 설정을 해놨어서 그거때문에 집주인한테 몇 번 전화했는데 이제 받지도 않음 이거 이런걸로 계약 파기 할 수 있나?",ok,3.16,3.16,3.16,0,0,없음
2,Q03,나 여기 계약했는데 귀신나와 이거 해지 사유 되나?,ok,22.13,8.83,2.67,5,5,없음
3,Q04,나는 자금이 부족한 상황이야. 그래서 전세자금 대출을 기반으로 계약을 진행하려고 해. 그러면 전세자금 대출 승인이 되어야 임대차 계약이 효력을 발휘하도록 하는 특약을 넣어도 될까?,ok,16.90,4.66,3.07,5,5,없음
4,Q05,내가 키우고 있는 고양이가 있는데 임대인이 특약을 넣자고 하네. 최대한 부담되지 않게 작성하려면 어떻게 해야 할까?,ok,16.72,2.46,2.14,5,2,없음
5,Q06,임대차 계약서 차임 뜻이 뭐야?,ok,6.27,2.55,2.42,5,2,없음
6,Q07,윗집이 밤마다 소음이 심한데 해결 방법 있을까?,ok,16.45,2.55,2.26,5,2,없음
7,Q08,공인중개사가 되려면 어떻게 준비해야돼?,ok,12.56,3.13,2.47,5,3,없음
8,Q09,"다른 사람이 집을 계약할까봐 보증금의 10프로는 먼저 지불했는데, 내가 계약을 못하게 되었을때 선지불했던 금액 다시 돌려받을 수 있을까?",ok,13.73,3.33,2.59,5,1,없음
9,Q10,혹시 자취생을 위한 주거비 지원 정책 좀 아는 거 있니?,ok,13.96,3.38,2.31,5,4,없음


전체 시간: P50 13.14초 / P95 22.89초 (n=14)


## 질문별 읽기
답변 전문은 모델이 출력한 그대로 표시합니다. 출처 링크가 있어도 주장을 실제로 뒷받침하는지는 별도로 검토해야 합니다. 각 질문 아래의 **검토 메모 Markdown 셀**을 직접 편집하세요.

In [3]:
def show_answer(question_id):
    """질문 ID를 받아 저장된 답변·시간·인용·검색 메타데이터를 표시합니다."""
    row = answers[question_id]
    display(Markdown("**질문**\n\n" + escape(row["question"])))
    display(summary.loc[summary["질문 ID"] == question_id].drop(columns=["질문"]).round(2))
    # 모델의 HTML을 실행하지 않으면서 Markdown 문단·목록을 읽기 쉽게 표시합니다.
    display(Markdown("**Before 답변 전문 — 모델 원문**\n\n" + escape(row["answer"])))
    links = []
    for citation in row["citations"]:
        title = escape(citation.get("title") or "제목 없음")
        url = citation.get("url") or ""
        if urlsplit(url).scheme in {"http", "https"}:
            links.append(f'<li><a href="{escape(url, quote=True)}">{title}</a></li>')
        else:
            links.append(f"<li>{title} — 사용 가능한 링크 없음</li>")
    display(HTML("<h4>응답에 붙은 인용 출처</h4><ul>" + "".join(links) + "</ul>" if links else "<p>인용 출처 없음</p>"))
    docs = row["trace"].get("retrieved_docs") or []
    display(Markdown("**검색 근거 목록** — 생성 모델에 최종 전달된 문맥 전체와는 구분합니다."))
    if docs:
        display(pd.DataFrame([{k: d.get(k) for k in ["title", "doc_type", "version", "source_url"]} for d in docs]))
    else:
        print("저장된 검색 근거 없음")
    print(f"원시 결과: {question_id}.json / usage: {row['usage']} / 비용: {row['cost']}")


## Q01

In [4]:
show_answer("Q01")

**질문**

월세 계약은 1년을 했는데 중간에 나가야할 일이 생겼는데 어떻게 해야할까?

,질문 ID,수집 상태,전체 시간(초),첫 토큰(초),전처리(초),검색 근거 수,인용 수,오류
0,Q01,ok,24.29,7.54,5.53,5,5,없음


**Before 답변 전문 — 모델 원문**

답변: 월세 계약이 1년으로 체결되었다고 하더라도, 중간에 계약을 종료해야 하는 상황이 발생한 경우 다음과 같은 절차를 고려할 수 있습니다:

1. **계약 당사자 간 합의**: 먼저 임대인과 임차인 간에 계약 종료에 대한 합의가 필요합니다. 합의가 이루어진다면, 합의해지의 조건과 임차보증금 반환 등에 대한 내용을 명확히 정리하는 것이 중요합니다. 합의해지는 계약의 효력을 장래에 향하여 소멸시키는 새로운 계약으로, 기존 계약의 이행이 시작된 후에 쌍방이 계약 실현 의사의 결여 또는 포기로 계약을 실현하지 않을 의사가 일치되어야 합니다 (대법원 2000. 3. 10. 선고 99다70884 판결 참조).

2. **조건 제시와 합의**: 임차인이 계약 종료를 요구할 경우, 임대인에게 계약 종료 조건을 제시해야 합니다. 예를 들어, 미지급 차임 정산, 임차보증금 반환 등에 대한 조건을 명확히 해야 합니다. 만약 임대인이 추가 조건을 제시한다면, 그 조건에 대한 합의가 이루어져야 합니다 (대법원 2018. 12. 27. 선고 2016다274270, 274287 판결 참조).

3. **법률적 절차**: 합의가 이루어지지 않는 경우, 법원에 계약 해지와 관련된 소송을 제기할 수 있습니다. 이때, 계약 해지의 필요성과 합리적인 조건을 입증해야 합니다. 특히, 계약 종료에 따른 법률관계 (예: 임차보증금 반환)가 중요한 관심사인 경우, 이러한 문제에 대한 명확한 해결 방안이 필요합니다 (대법원 1994. 9. 13. 선고 94다17093 판결 참조).

4. **중재 또는 조정**: 분쟁 해결을 위해 중재나 조정 절차를 활용할 수도 있습니다. 이는 소송보다 빠르고 비용 효율적인 방법일 수 있습니다.

**주의사항**:
- 계약 해지 시, 계약 기간 중 해지에 따른 손해배상금이나 차임 연체료 등이 발생할 수 있으므로, 이러한 비용에 대한 명확한 합의가 필요합니다.
- 계약 해지와 관련된 모든 의사소통은 문서화하여 추후 분쟁을 방지하는 것이 좋습니다.

이러한 절차를 통해 계약을 중도에 종료하고, 관련 법률적 문제를 해결할 수 있습니다. 하지만 구체적인 상황에 따라 법률 자문을 받는 것이 가장 안전한 방법입니다.

---
**관련 법령/판례:**
- [판례] [대법원 2020다279456, 279463 건물명도(인도)·임대차보증금 2020다279456, 279463](https://www.law.go.kr/LSW/precInfoP.do?precSeq=219161)
- [법령] [민법 제635조](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%AF%BC%EB%B2%95/%EC%A0%9C635%EC%A1%B0)
- [법령] [주택임대차보호법 제6조의2](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A3%BC%ED%83%9D%EC%9E%84%EB%8C%80%EC%B0%A8%EB%B3%B4%ED%98%B8%EB%B2%95/%EC%A0%9C6%EC%A1%B0%EC%9D%982)
- [판례] [대법원 2017다260636 임대차보증금 2017다260636](https://www.law.go.kr/LSW/precInfoP.do?precSeq=211785)
- [판례] [대법원 2023다236566, 236573 임차보증금반환·차임연체료및손해배상금 2023다236566, 236573](https://www.law.go.kr/LSW/precInfoP.do?precSeq=239521)

**검색 근거 목록** — 생성 모델에 최종 전달된 문맥 전체와는 구분합니다.

,title,doc_type,version,source_url
0,"대법원 2020다279456, 279463 건물명도(인도)·임대차보증금",판례,"판례|2020다279456, 279463",https://www.law.go.kr/LSW/precInfoP.do?precSeq...
1,민법,법령,민법|제635조@2026-03-17,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%A...
2,주택임대차보호법,법령,주택임대차보호법|제6조의2@2008-03-21,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...
3,대법원 2017다260636 임대차보증금,판례,판례|2017다260636,https://www.law.go.kr/LSW/precInfoP.do?precSeq...
4,"대법원 2023다236566, 236573 임차보증금반환·차임연체료및손해배상금",판례,"판례|2023다236566, 236573",https://www.law.go.kr/LSW/precInfoP.do?precSeq...


원시 결과: Q01.json / usage: unknown / 비용: unknown


### 검토 메모 — 수정 필요

- **질문의 조건을 제대로 반영했는지:** ‘1년 약정 중 개인 사정으로 퇴거’라는 상황에서 합의해지를 우선 제시한 방향은 맞다. 그러나 최초 계약인지 묵시적 갱신인지 확인하지 않아 적용할 해지 규칙을 구분하지 못했다. 주택임대차보호법 제4조의 2년 보호는 1년 도중 언제든 나갈 권리를 뜻하지 않는다.
- **답변과 근거·인용이 일치하는지:** 2023다236566·236573은 합의해지의 조건과 보증금 정산에 관한 설명을 뒷받침한다. 반면 2020다279456·279463은 차임 연체 해지통보의 해석, 2017다260636은 재개발에 따른 해지 사건이다. 개인 사정으로 중도해지를 강제할 근거가 아니다. 민법 제635조는 기간을 정하지 않은 임대차 규정이며, 주택임대차보호법 제6조의2도 최초 1년 계약에 곧바로 적용할 수 없다.
- **필요한 추가 질문·예외가 있는지:** 최초 계약/갱신 여부, 시작·만료일, 원하는 퇴거일, 중도해지 특약, 퇴거 사유를 묻는다. 묵시적 갱신 또는 갱신요구권 행사에 따른 갱신이면 통지 도달 후 3개월 규칙을 검토한다. 일반적인 합의 갱신까지 일률적으로 포함하지 않는다.
- **설명이 쉽고 다음 행동을 알 수 있는지:** 판례 문장이 길어 정작 ‘오늘 무엇을 할지’가 흐려진다. 계약서 확인 → 임대인에게 희망 종료일 제안 → 월세 정산 종료일·보증금 반환일·열쇠 인도·새 임차인 모집 조건을 문자 또는 합의서로 확정하는 순서가 필요하다. 중개보수 부담은 중도퇴거라는 이유만으로 자동 확정하지 말고 약정·협상 사항으로 설명한다.
- **심각한 오류·수정 의견:** ‘해지의 필요성과 합리적인 조건을 입증’하면 소송으로 종료할 수 있다는 설명은 삭제한다. 법정·약정 해지권이 없으면 개인 사정만으로 종료를 강제하기 어렵고, 이사만 해도 월세 의무가 끝나지 않을 수 있음을 먼저 알려야 한다. 대체 문장: ‘첫 계약 기간 중이라면 중도해지 특약이나 별도 사유가 없는 한 집주인과 종료 조건을 합의하는 것이 우선입니다.’

점수: **2/5 — 공인중개사 자문을 반영한 확정 정성 점수**

**보류 사항:** 사실관계·계약 연도·적용 판본 등 미확정 조건을 유지합니다. 필요한 추가 질문과 조건부 설명은 검토할 수 있으나 특정 조건의 검색 점수는 확정 전 보류합니다.

**검토에 사용한 공식 자료** (2026-10-18 확인; 원래 답변의 검색 결과와 별도):

- [합의해지의 성립과 정산 조건](https://www.law.go.kr/LSW/precInfoP.do?precSeq=239521)
- [기간 없는 임대차와 구분할 조문](https://www.law.go.kr/LSW/lsLinkCommonInfo.do?chrClsCd=010202&lsJoLnkSeq=1026989829)

**공통 채점·검토 범위:** 2026-10-19 사용자가 고정·확정한 신뢰·활용 가능성 5점 척도(trust-usability-v1)를 사용합니다. 공인중개사 자문을 반영한 평가이며 세부 기준은 evaluation_protocol.md에 기록했습니다. 1점=핵심 결론 오류 또는 실질적 안내 부재, 2점=일부 방향만 맞고 핵심 판단·대응 누락, 3점=핵심 방향은 맞으나 적용 조건·근거·행동 안내 보완 필요, 4점=경미한 수정으로 활용 가능, 5점=질문 범위에서 정확하고 실행 가능하며 근거가 확인됨입니다. 항목별 가중 합산이 아닌 종합 정성 점수입니다. 사용자가 공인중개사 자문을 받아 중대한 오류와 평가 확정 여부를 작성했다고 확인했습니다. 정식 블라인드 평가나 별도의 공인 인증을 뜻하지 않습니다. 검색 문서 본문·최종 전달 문맥이 첨부되지 않아 faithfulness 및 검색 지표(precision/recall 등)는 확정하지 않습니다. 공개 공식 자료의 확인 가능한 내용과 인용 링크의 관련성, 답변 자체의 조건 반영·실무 활용성을 구분해 검토했습니다. URL에 붙은 날짜 메타데이터만으로 적용 판본을 확정하지 않았으며, 각 사건의 실제 조건은 추가 확인이 필요합니다. 특약 예시는 협의용 제안입니다.


## Q02

In [5]:
show_answer("Q02")

**질문**

내가 지금 전세금 3억정도의 집에 살고있는데, 전세 계약 만료일이 1년 정도 남았단 말이야? 근데 특약으로 전세권 설정을 해놨어서 그거때문에 집주인한테 몇 번 전화했는데 이제 받지도 않음 이거 이런걸로 계약 파기 할 수 있나?

,질문 ID,수집 상태,전체 시간(초),첫 토큰(초),전처리(초),검색 근거 수,인용 수,오류
1,Q02,ok,3.16,3.16,3.16,0,0,없음


**Before 답변 전문 — 모델 원문**

죄송합니다, 해당 질문에 대한 정확한 답변을 찾지 못했습니다. 질문을 좀 더 구체적으로 입력해주시겠어요?

**검색 근거 목록** — 생성 모델에 최종 전달된 문맥 전체와는 구분합니다.

저장된 검색 근거 없음
원시 결과: Q02.json / usage: unknown / 비용: unknown


### 검토 메모 — 수정 필요

- **질문의 조건을 제대로 반영했는지:** 전세금 약 3억 원, 만료까지 약 1년, 전세권 설정 특약, 연락 두절이라는 조건을 전혀 활용하지 못했다. ‘전세권 설정을 해놨다’가 등기 완료인지 설정 약속만 했다는 뜻인지가 핵심인데, 이를 특정한 질문으로 되묻지 않았다.
- **답변과 근거·인용이 일치하는지:** 검색·인용이 모두 없어 사실과 법률의 일치 여부를 평가할 근거가 없다. 근거 없이 계약 파기를 단정하지 않은 점은 적절하지만, 검색 실패와 법적 대응 불가능은 다른 문제다.
- **필요한 추가 질문·예외가 있는지:** ① 등기사항증명서에 전세권이 실제 기재되어 있는지 ② 특약 원문과 설정 기한 ③ 잔금 지급·입주 여부 ④ 임대인이 제출하지 않은 서류나 거절한 협조가 무엇인지 확인한다. 전입신고·확정일자와 전세권설정등기는 서로 다르다고 풀어준다.
- **설명이 쉽고 다음 행동을 알 수 있는지:** ‘구체적으로 입력’만 요청하면 사용자는 무엇을 더 말할지 모른다. ‘특약 문구와 설정 기한을 개인정보를 가리고 알려주세요. 등기부에 전세권이 이미 올라가 있나요?’라고 묻고, 계약서·등기부·문자·통화 이력을 확보하도록 안내해야 한다.
- **심각한 오류·수정 의견:** 연락을 받지 않는다는 사정만으로 즉시 해지된다고 할 수 없다. 설정 약속을 이행하지 않았다면 이행기 도래, 중요한 의무인지, 상당한 기간을 정한 이행 요구와 불이행 여부 등을 검토해야 한다. 내용증명은 이행 요구 내용과 발송 사실 등을 남기는 수단이며, 발송만으로 계약 종료나 반환이 보장되지는 않는다. 이미 등기가 완료된 경우에는 어떤 별도 의무가 이행되지 않았는지부터 다시 확인한다.

점수: **1/5 — 공인중개사 자문을 반영한 확정 정성 점수**

**검토에 사용한 공식 자료** (2026-10-18 확인; 원래 답변의 검색 결과와 별도):

- [이행지체와 이행 최고: 민법 제544조](https://www.law.go.kr/LSW/lsInfoP.do?lsiSeq=67272)


## Q03

In [6]:
show_answer("Q03")

**질문**

나 여기 계약했는데 귀신나와 이거 해지 사유 되나?

,질문 ID,수집 상태,전체 시간(초),첫 토큰(초),전처리(초),검색 근거 수,인용 수,오류
2,Q03,ok,22.13,8.83,2.67,5,5,없음


**Before 답변 전문 — 모델 원문**

답변: 임대차 계약을 해지할 수 있는 사유는 다양하며, 주택임대차보호법 및 관련 판례에 따르면 임차인이 계약을 해지할 수 있는 상황은 다음과 같습니다:

1. **임대인의 책임 있는 사유로 인한 임대차 목적물의 훼손 또는 하자**: 임대인이 임대차 목적물의 유지보수 의무를 이행하지 않아 임차인이 정상적으로 사용할 수 없게 된 경우, 계약 해지가 가능할 수 있습니다. 예를 들어, 건물의 구조적 문제나 안전상의 이유로 인해 거주가 불가능해진 경우가 여기에 해당할 수 있습니다. 하지만 &#x27;귀신이 나온다&#x27;는 사유는 일반적으로 법적으로 인정되는 구체적인 하자나 훼손 사유로 보기 어렵습니다.

2. **임대인의 귀책사유**: 임대인이 계약상의 의무를 이행하지 않거나 불법적인 행위를 한 경우 계약 해지가 가능합니다. 그러나 &#x27;귀신이 나온다&#x27;는 사유는 주관적이고 증명하기 어려운 사항으로, 일반적인 법적 판단 기준에 부합하지 않을 가능성이 큽니다.

3. **계약의 합의해지**: 임대인과 임차인 간에 합의가 이루어진 경우 계약을 해지할 수 있습니다. 하지만 합의해지 시에는 원상회복에 관한 약정이 필요하며, 특히 임대차 계약에서 이미 지급된 임차보증금의 반환에 관하여는 합의가 이루어지지 않은 채 계약만 해지되는 것은 경험칙에 비춰 이례적입니다.

따라서, &#x27;귀신이 나온다&#x27;는 사유만으로는 일반적으로 임대차 계약의 법적 해지 사유로 인정받기 어렵습니다. 구체적인 하자나 임대인의 책임 있는 사유를 명확히 입증할 수 있어야 합니다. 만약 이러한 상황이 발생했다면, 임대인과의 합의를 통해 문제를 해결하거나 법적 조언을 구하는 것이 바람직합니다.

---
**관련 법령/판례:**
- [판례] [대법원 92다4130, 92다4147 임차보증금반환등 92다4130, 92다4147](https://www.law.go.kr/LSW/precInfoP.do?precSeq=203156)
- [판례] [대법원 2023다236566, 236573 임차보증금반환·차임연체료및손해배상금 2023다236566, 236573](https://www.law.go.kr/LSW/precInfoP.do?precSeq=239521)
- [판례] [서울중앙지방법원 2008가합17965 임차보증금반환 2008가합17965](https://www.law.go.kr/LSW/precInfoP.do?precSeq=144159)
- [판례] [서울고등법원 2022나2043727 임대차보증금등반환청구의소 2022나2043727](https://www.law.go.kr/LSW/precInfoP.do?precSeq=238469)
- [판례] [대법원 2001다30285 임대차보증금 2001다30285](https://www.law.go.kr/LSW/precInfoP.do?precSeq=81232)

**검색 근거 목록** — 생성 모델에 최종 전달된 문맥 전체와는 구분합니다.

,title,doc_type,version,source_url
0,"대법원 92다4130, 92다4147 임차보증금반환등",판례,"판례|92다4130, 92다4147",https://www.law.go.kr/LSW/precInfoP.do?precSeq...
1,"대법원 2023다236566, 236573 임차보증금반환·차임연체료및손해배상금",판례,"판례|2023다236566, 236573",https://www.law.go.kr/LSW/precInfoP.do?precSeq...
2,서울중앙지방법원 2008가합17965 임차보증금반환,판례,판례|2008가합17965,https://www.law.go.kr/LSW/precInfoP.do?precSeq...
3,서울고등법원 2022나2043727 임대차보증금등반환청구의소,판례,판례|2022나2043727,https://www.law.go.kr/LSW/precInfoP.do?precSeq...
4,대법원 2001다30285 임대차보증금,판례,판례|2001다30285,https://www.law.go.kr/LSW/precInfoP.do?precSeq...


원시 결과: Q03.json / usage: unknown / 비용: unknown


### 검토 메모 — 수정 필요

- **질문의 조건을 제대로 반영했는지:** ‘귀신이 나온다’는 주장만으로 법정 해지 사유를 인정하기 어렵다는 결론은 타당한 방향이다. 다만 실제 이상 소음·설비 하자를 말하는지, 사건 이력을 뒤늦게 알았다는 뜻인지 구분하지 않고 일반적인 해지 사유를 반복했다.
- **답변과 근거·인용이 일치하는지:** 2023다236566·236573은 합의해지와 보증금 정산에 관한 부분을 지원한다. 2008가합17965는 병원 화재 후 수선의무 지체 사건으로 귀신이나 심리적 불쾌감에 관한 직접 판례가 아니다. 나열한 판례만으로 ‘귀신 관련 판례가 있다’고 읽히지 않게 해야 한다. 나머지 판례가 각각 어느 주장에 대응하는지는 확인이 필요하다.
- **필요한 추가 질문·예외가 있는지:** 무엇을 직접 경험했는지, 거주를 방해하는 소음·진동·전기·가스 이상이 있는지, 계약 전 어떤 설명을 들었는지, 확인 가능한 사건 이력과 허위 설명 자료가 있는지 묻는다. 사건 이력이 있다 해도 자동 해지로 단정하지 않고 고지·기망·계약 체결에 미친 영향 등을 개별 검토한다.
- **설명이 쉽고 다음 행동을 알 수 있는지:** 사용자의 불안을 인정하면서 확인 가능한 문제로 좁혀야 한다. ‘어떤 현상이 있었나요? 소음·설비 문제라면 시간과 상태를 기록해 집주인에게 점검을 요청하세요’가 유용하다. 객관적 사유가 없다면 합의해지와 보증금 반환일·정산 조건 협의로 연결한다.
- **심각한 오류·수정 의견:** ‘임대인에게 책임 있는 훼손’만 수선 대상인 것처럼 제한하지 않는다. 실제 인용된 화재 판결도 임대인의 귀책사유 없는 훼손에 대한 수선의무를 원칙적으로 인정한다. 또한 ‘원상회복 약정이 필요’는 모든 합의해지의 일률적 법정 요건처럼 쓰지 말고, 쟁점이 된 정산 조건에 합의해야 한다는 취지로 수정한다.

점수: **3/5 — 공인중개사 자문을 반영한 확정 정성 점수**

**검토에 사용한 공식 자료** (2026-10-18 확인; 원래 답변의 검색 결과와 별도):

- [화재와 수선의무 사건: 사실관계 구별 필요](https://www.law.go.kr/LSW/precInfoP.do?precSeq=144159)
- [합의해지의 조건](https://www.law.go.kr/LSW/precInfoP.do?precSeq=239521)


## Q04

In [7]:
show_answer("Q04")

**질문**

나는 자금이 부족한 상황이야. 그래서 전세자금 대출을 기반으로 계약을 진행하려고 해. 그러면 전세자금 대출 승인이 되어야 임대차 계약이 효력을 발휘하도록 하는 특약을 넣어도 될까?

,질문 ID,수집 상태,전체 시간(초),첫 토큰(초),전처리(초),검색 근거 수,인용 수,오류
3,Q04,ok,16.9,4.66,3.07,5,5,없음


**Before 답변 전문 — 모델 원문**

답변: 자금 부족으로 인해 전세자금 대출을 기반으로 임대차 계약을 진행하려는 경우, 대출 승인을 임대차 계약의 효력 발생 조건으로 명시하는 특약을 넣는 것은 가능합니다. 그러나 이는 계약 당사자 간의 합의 사항이며, 법적으로 명시적으로 허용되는 방식입니다. 

특약을 통해 대출 승인을 계약 효력 발생의 전제 조건으로 설정할 수 있지만, 몇 가지 중요한 점을 고려해야 합니다:

1. **명확한 합의**: 대출 승인 조건이 계약 효력 발생에 필수적임을 명확히 계약서에 기재해야 합니다. 이는 양 당사자가 동의한 내용이어야 하며, 불필요한 오해를 방지하기 위해 구체적으로 명시하는 것이 좋습니다.

2. **대출 조건**: 대출 승인 조건(예: 대출 승인일, 대출 금액 등)을 구체적으로 명시해야 합니다. 이는 계약 효력 발생 시점을 명확히 하는 데 도움이 됩니다.

3. **위약금 및 책임**: 대출 승인이 이루어지지 않을 경우의 책임과 위약금 등에 대한 조항도 함께 명시하는 것이 바람직합니다. 이를 통해 양 당사자의 권리와 의무를 명확히 할 수 있습니다.

예를 들어, 계약서에 다음과 같은 내용을 포함시킬 수 있습니다:

- **대출 승인 조건**: &quot;임대차 계약의 효력은 임차인이 지정된 금융기관으로부터 전세자금 대출을 승인받은 날로부터 발생한다.&quot;
- **대출 승인 미확시**: &quot;대출 승인이 이루어지지 않을 경우, 임차인은 계약 체결 후 7일 이내에 계약을 해지할 수 있으며, 이 경우 임차인은 계약금을 반환받으며 임대인은 계약금을 반환해야 한다.&quot;

이러한 특약은 계약 당사자 간의 합의에 따라 유연하게 조정될 수 있으나, 법적 효력을 갖추기 위해서는 명확하고 구체적인 표현이 필요합니다.

---
**관련 법령/판례:**
- [판례] [대법원 88다카6228 전세금 88다카6228](https://www.law.go.kr/LSW/precInfoP.do?precSeq=105012)
- [판례] [대법원 83다카1847 전세금반환 83다카1847](https://www.law.go.kr/LSW/precInfoP.do?precSeq=206547)
- [법령해석례] [법령해석례 20-0532 민원인 - 「민간임대주택에 관한 특별법」 제44조제1항에 따른 최초 임대료에 대한 「주택임대차보호법」 제6조의3제3항 단서 적용 여부(법률 제16386호 「민간임대주택에 관한 특별법」 부칙 제4조 등 관련) 20-0532](https://www.law.go.kr/LSW/expcInfoP.do?expcSeq=329097)
- [법령해석례] [법령해석례 20-0528 민원인 - 「민간임대주택에 관한 특별법」 제44조제1항에 따른 최초 임대료에 대한 「주택임대차보호법」 제6조의3제3항 단서 적용 여부(법률 제16386호 「민간임대주택에 관한 특별법」 부칙 제4조 등 관련) 20-0528](https://www.law.go.kr/LSW/expcInfoP.do?expcSeq=331405)
- [판례] [대법원 2023다202228 임대차보증금반환 2023다202228](https://www.law.go.kr/LSW/precInfoP.do?precSeq=241081)

**검색 근거 목록** — 생성 모델에 최종 전달된 문맥 전체와는 구분합니다.

,title,doc_type,version,source_url
0,대법원 88다카6228 전세금,판례,판례|88다카6228,https://www.law.go.kr/LSW/precInfoP.do?precSeq...
1,대법원 83다카1847 전세금반환,판례,판례|83다카1847,https://www.law.go.kr/LSW/precInfoP.do?precSeq...
2,법령해석례 20-0532 민원인 - 「민간임대주택에 관한 특별법」 제44조제1항에 ...,법령해석례,해석례|20-0532,https://www.law.go.kr/LSW/expcInfoP.do?expcSeq...
3,법령해석례 20-0528 민원인 - 「민간임대주택에 관한 특별법」 제44조제1항에 ...,법령해석례,해석례|20-0528,https://www.law.go.kr/LSW/expcInfoP.do?expcSeq...
4,대법원 2023다202228 임대차보증금반환,판례,판례|2023다202228,https://www.law.go.kr/LSW/precInfoP.do?precSeq...


원시 결과: Q04.json / usage: unknown / 비용: unknown


### 검토 메모 — 수정 필요

- **질문의 조건을 제대로 반영했는지:** 대출 승인을 조건으로 하는 특약을 당사자 합의로 정할 수 있다는 핵심은 반영했다. 그러나 사용자의 실제 목적은 ‘자금 조달에 실패했을 때 계약금을 잃지 않는 것’인데, 승인 여부와 실제 대출 실행·필요 금액 확보를 구분하지 않았다.
- **답변과 근거·인용이 일치하는지:** 민법 제147조의 정지조건이 직접 검토할 근거다. 인용한 88다카6228은 부동산 교환·매매에서 채무인수의 효력 시점 사건이며, 2023다202228은 보증금반환채권 근질권 사건이다. 대출 불승인 시 임차인에게 계약금 반환권이 당연히 생긴다는 근거로 쓸 수 없다. 임대료 관련 해석례도 해당 특약의 직접 근거가 아니다.
- **필요한 추가 질문·예외가 있는지:** 대출 상품·필요 최소액·잔금일·심사 예상기간·본인 신용과 주택 권리관계 중 어떤 불승인 사유까지 보장받을지 확인한다. 보증기관 심사와 은행 실행 조건도 확인한다. 계약 전체의 효력을 미루는 방식이 은행 제출·심사 절차와 맞는지 먼저 문의하도록 한다.
- **설명이 쉽고 다음 행동을 알 수 있는지:** 예시의 ‘계약 후 7일’은 은행 심사가 끝나기 전에 지나갈 수 있다. 승인 기한과 잔금일, 통지 기한·방법, 불승인 증빙, 계약금 반환 기한, 위약금 배제, 임대인의 서류 협조를 묶어서 제시해야 실무에 쓸 수 있다.
- **심각한 오류·수정 의견:** 정지조건과 ‘유효한 계약을 불승인 시 해제하는 약정’을 혼합했다. 둘 중 구조를 정해 일관되게 작성한다. 협의용 예: ‘임차인이 ○월 ○일까지 ○원 이상 전세대출을 승인받지 못한 경우, [합의한 적용 사유]에 한해 ○일까지 증빙을 첨부하여 해제할 수 있다. 임대인은 통지 도달 후 ○영업일 내 기수령 계약금 전액을 반환하며 위약금은 청구하지 않는다.’ 적용 사유·심사 지연 처리·실행 실패 포함 여부를 빈칸 없이 합의해야 하며 보편적인 확정 문안으로 제시하면 안 된다.

점수: **2/5 — 공인중개사 자문을 반영한 확정 정성 점수**

**검토에 사용한 공식 자료** (2026-10-18 확인; 원래 답변의 검색 결과와 별도):

- [민법 제147조: 조건 성취와 효력](https://law.go.kr/LSW/lsInfoP.do?ancYnChk=0&chrClsCd=010202&efYd=20230628&joNo=010900&lsiSeq=246569&urlMode=lsInfoP)
- [88다카6228의 실제 쟁점](https://www.law.go.kr/LSW/precInfoP.do?precSeq=105012)


## Q05

In [8]:
show_answer("Q05")

**질문**

내가 키우고 있는 고양이가 있는데 임대인이 특약을 넣자고 하네. 최대한 부담되지 않게 작성하려면 어떻게 해야 할까?

,질문 ID,수집 상태,전체 시간(초),첫 토큰(초),전처리(초),검색 근거 수,인용 수,오류
4,Q05,ok,16.72,2.46,2.14,5,2,없음


**Before 답변 전문 — 모델 원문**

답변: 고양이와 관련된 특약 조항을 작성할 때는 임대인과 임차인 간의 합의를 바탕으로 명확하고 합리적인 조건을 설정하는 것이 중요합니다. 다음과 같은 요소들을 고려하여 특약을 구성할 수 있습니다:

1. **책임 범위 명확화**:
   - 고양이의 행동으로 인한 손해 (예: 가구 손상, 벽 파손 등)에 대한 책임을 명확히 정의합니다. 예를 들어, &quot;임차인은 고양이의 행동으로 인한 재산 피해에 대해 책임을 진다&quot;와 같이 작성할 수 있습니다.
   - **출처**: 일반적으로 민법 제750조 (불법행위에 대한 책임)를 참고하여 책임의 범위를 설정할 수 있습니다.

2. **관리 및 청소 책임**:
   - 고양이의 배설물 처리, 털 관리 등 일상적인 관리 책임을 명시합니다. 예를 들어, &quot;임차인은 고양이의 배설물 처리 및 털 관리에 대한 책임을 진다&quot;와 같이 작성할 수 있습니다.

3. **보험 가입**:
   - 임차인이 고양이 관련 보험에 가입하도록 요구할 수 있습니다. 예를 들어, &quot;임차인은 고양이 관련 손해를 보상하기 위한 보험에 가입해야 한다&quot;와 같이 명시할 수 있습니다.

4. **사전 동의 및 통지**:
   - 고양이를 키우기 전에 임대인의 사전 동의를 받는 조항을 포함시킵니다. 예를 들어, &quot;임차인은 고양이를 키우기 전에 임대인의 서면 동의를 받아야 하며, 고양이의 건강 상태 등에 대한 정보를 제공해야 한다&quot;와 같이 작성할 수 있습니다.

5. **계약 종료 시 처리**:
   - 계약 종료 시 고양이의 관리 및 반환 조건을 명시합니다. 예를 들어, &quot;계약 종료 시 임차인은 고양이를 반환하거나 임대인이 지정한 장소로 이동시켜야 한다&quot;와 같이 작성할 수 있습니다.

이러한 내용들을 포함하여 특약을 작성하면 양측 모두에게 부담을 최소화하면서도 명확한 책임과 기대치를 설정할 수 있습니다. 특약 조항은 양측이 합의한 내용을 바탕으로 작성되어야 하며, 필요하다면 법률 전문가의 조언을 구하는 것도 좋은 방법입니다.

**출처**:
- 민법 제750조 (불법행위에 대한 책임)

---
**관련 법령/판례:**
- [법령] [민법 제510조](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%AF%BC%EB%B2%95/%EC%A0%9C510%EC%A1%B0)
- [법령] [부동산 거래신고 등에 관한 법률 시행령 제6조](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B6%80%EB%8F%99%EC%82%B0%20%EA%B1%B0%EB%9E%98%EC%8B%A0%EA%B3%A0%20%EB%93%B1%EC%97%90%20%EA%B4%80%ED%95%9C%20%EB%B2%95%EB%A5%A0%20%EC%8B%9C%ED%96%89%EB%A0%B9/%EC%A0%9C6%EC%A1%B0)

**검색 근거 목록** — 생성 모델에 최종 전달된 문맥 전체와는 구분합니다.

,title,doc_type,version,source_url
0,민법,법령,민법|제510조@2026-03-17,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%A...
1,부동산 거래신고 등에 관한 법률 시행령,법령,부동산 거래신고 등에 관한 법률 시행령|제6조@2017-01-20,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B...
2,부동산 거래신고 등에 관한 법률 시행령,법령,부동산 거래신고 등에 관한 법률 시행령|제5조@2020-02-21,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B...
3,부동산 거래신고 등에 관한 법률 시행령,법령,부동산 거래신고 등에 관한 법률 시행령|제5조@2017-01-20,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B...
4,부동산 거래신고 등에 관한 법률 시행령,법령,부동산 거래신고 등에 관한 법률 시행령|제6조@2023-10-04,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B...


원시 결과: Q05.json / usage: unknown / 비용: unknown


### 검토 메모 — 수정 필요 — 우선 수정

- **질문의 조건을 제대로 반영했는지:** ‘이미 고양이를 키우며 임차인 부담을 줄이고 싶다’는 요청과 반대로 보험 가입·건강 정보 제공·이동 의무를 추가했다. 책임 항목을 많이 넣는 것을 부담 최소화로 착각한 답변이다.
- **답변과 근거·인용이 일치하는지:** 본문의 민법 제750조는 일반 불법행위 책임 규정일 뿐 보험 가입이나 고양이 인도·이동 의무의 근거가 아니다. 실제 연결한 민법 제510조는 배서 방식 규정으로 무관하다. 부동산 거래신고 시행령도 반려묘 특약의 직접 근거가 아니다. 본문 출처와 링크 목록부터 불일치한다.
- **필요한 추가 질문·예외가 있는지:** 임대인이 요구한 특약 원문, 사육 동의 여부, 마릿수, 기존 벽지·바닥 상태, 옵션 가구, 관리규약상 제한을 확인한다. 보험은 해당 손해가 실제 보장되는 상품인지 확인 없이 의무로 추천하지 않는다.
- **설명이 쉽고 다음 행동을 알 수 있는지:** ‘임대인은 현재 고양이 ○마리 사육에 동의한다’를 먼저 명시하고, 입주 시 사진을 함께 남기도록 한다. 협의용 문안: ‘반려묘로 인해 발생한 것으로 확인되는 훼손은 기존 상태·노후도를 고려해 필요한 범위의 합리적인 복구비를 협의한다. 통상 사용·자연 마모는 제외하며, 정액 벌금이나 훼손과 무관한 일괄 도배·교체비는 부과하지 않는다.’ 이는 부담을 줄이기 위한 제안이며 자동 적용되는 법정 문구는 아니다.
- **심각한 오류·수정 의견:** ‘고양이를 반환하거나 임대인이 지정한 장소로 이동’은 삭제해야 한다. 임대차 목적물인 집과 임차인의 반려동물을 혼동한 문구다. 사소한 민원만으로 즉시 퇴거·보증금 몰수·무제한 배상하게 하는 조항도 피하도록 안내하고, 손해의 원인·범위·금액을 확인할 증빙과 시정 협의 절차를 넣는다.

점수: **1/5 — 공인중개사 자문을 반영한 확정 정성 점수**

**검토에 사용한 공식 자료** (2026-10-18 확인; 원래 답변의 검색 결과와 별도):

- [민법 제510조의 실제 내용: 배서 방식](https://www.law.go.kr/LSW/lsLawLinkInfo.do?chrClsCd=010202&lsId=001706&lsJoLnkSeq=900410016&print=print)
- [민법 제750조: 고의·과실과 손해배상](https://www.law.go.kr/lsInfoP.do?lsiSeq=122983)


## Q06

In [9]:
show_answer("Q06")

**질문**

임대차 계약서 차임 뜻이 뭐야?

,질문 ID,수집 상태,전체 시간(초),첫 토큰(초),전처리(초),검색 근거 수,인용 수,오류
5,Q06,ok,6.27,2.55,2.42,5,2,없음


**Before 답변 전문 — 모델 원문**

답변: 차임은 임대차 계약에서 임대인에게 임차인이 지급하는 대가를 의미합니다. 이는 주로 주택이나 부동산을 사용하고 수익을 얻는 대가로 지불되는 금전적 보상입니다. 주택임대차보호법 및 관련 시행령에 따르면, 차임의 증액은 약정한 차임의 20분의 1을 초과할 수 없으며, 증액 후 1년 이내에는 추가 증액을 청구할 수 없습니다 (출처: 주택임대차보호법 시행령 제2조, 제8조). 따라서 차임은 계약에서 임차인의 사용 권한에 대한 금전적 보상으로 이해할 수 있습니다.

---
**관련 법령/판례:**
- [법령] [민법 제618조](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%AF%BC%EB%B2%95/%EC%A0%9C618%EC%A1%B0)
- [법령] [주택임대차보호법 시행령 제8조](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A3%BC%ED%83%9D%EC%9E%84%EB%8C%80%EC%B0%A8%EB%B3%B4%ED%98%B8%EB%B2%95%20%EC%8B%9C%ED%96%89%EB%A0%B9/%EC%A0%9C8%EC%A1%B0)

**검색 근거 목록** — 생성 모델에 최종 전달된 문맥 전체와는 구분합니다.

,title,doc_type,version,source_url
0,민법,법령,민법|제618조@2026-03-17,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%A...
1,주택임대차보호법 시행령,법령,주택임대차보호법 시행령|제8조@2014-01-01,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...
2,주택임대차보호법 시행령,법령,주택임대차보호법 시행령|제2조@2008-08-21,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...
3,주택임대차보호법 시행령,법령,주택임대차보호법 시행령|제2조@2012-01-06,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...
4,주택임대차보호법 시행령,법령,주택임대차보호법 시행령|제2조@2013-03-23,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...


원시 결과: Q06.json / usage: unknown / 비용: unknown


### 검토 메모 — 수정 필요

- **질문의 조건을 제대로 반영했는지:** 차임이 사용·수익의 대가라는 정의는 맞다. 다만 ‘차임 뜻’만 묻는 사용자에게 인상률과 제한 기간까지 설명해 단순한 질문의 난도를 불필요하게 높였다.
- **답변과 근거·인용이 일치하는지:** 민법 제618조는 정의에 부합한다. 검색 목록은 시행령 제8조@2014-01-01과 제2조의 과거 판본들을 섞었고, 버전을 고정하지 않은 링크를 제시했다. 과거 시행령에 있던 증액 규정을 오늘의 근거처럼 읽게 하면 안 된다. 현재 증액청구 제한은 주택임대차보호법 제7조로 확인하는 편이 정확하다.
- **필요한 추가 질문·예외가 있는지:** 용어 설명 자체에 추가 질문은 필요 없다. 사용자가 계약서 금액을 헷갈려 할 때만 보증금·월세·관리비 항목을 개인정보 없이 확인한다.
- **설명이 쉽고 다음 행동을 알 수 있는지:** ‘차임은 집을 빌려 쓰는 대가로 내는 돈으로, 월세 계약에서는 보통 매달 내는 월세를 뜻해요. 예를 들어 보증금 1,000만 원/월세 50만 원이면 월 차임은 50만 원이에요. 보증금과 별도 표시된 관리비는 따로 확인하세요’ 정도면 충분하다.
- **심각한 오류·수정 의견:** 증액 설명은 삭제하거나 후속 질문으로 분리한다. 유지한다면 ‘모든 새 계약의 임대료가 무조건 5% 이내’라는 뜻으로 넓히지 말고, 증액청구·갱신의 적용 맥락을 명시한다. 법 제7조상 1년 제한도 증액 후뿐 아니라 계약 체결 후를 포함하는데 현재 설명은 이를 빠뜨렸다.

점수: **3/5 — 공인중개사 자문을 반영한 확정 정성 점수**

**검토에 사용한 공식 자료** (2026-10-18 확인; 원래 답변의 검색 결과와 별도):

- [임대차와 차임: 민법 제618조](https://www.law.go.kr/LSW/lsInfoP.do?lsiSeq=60398)
- [현행 증액청구 제한: 주택임대차보호법 제7조](https://www.law.go.kr/LSW/lsLinkCommonInfo.do?chrClsCd=010202&lsJoLnkSeq=1031734231)


## Q07

In [10]:
show_answer("Q07")

**질문**

윗집이 밤마다 소음이 심한데 해결 방법 있을까?

,질문 ID,수집 상태,전체 시간(초),첫 토큰(초),전처리(초),검색 근거 수,인용 수,오류
6,Q07,ok,16.45,2.55,2.26,5,2,없음


**Before 답변 전문 — 모델 원문**

답변: 윗집의 밤 시간대 소음 문제 해결을 위해 다음과 같은 방법들을 고려해 볼 수 있습니다:

1. **관리주체에게 신고**: 공동주택관리법 제20조에 따라, 층간소음으로 피해를 입었다면 관리주체에게 층간소음 발생 사실을 알려야 합니다. 관리주체는 사실관계 확인을 위해 필요한 조사를 진행할 수 있습니다.
   - **출처**: 공동주택관리법 제20조 (법령 조문번호: 제20조)

2. **권고 및 조정 요청**: 관리주체가 윗집에 층간소음 발생 중단이나 소음 차단 조치를 권고하도록 요청할 수 있습니다. 권고에도 불구하고 소음이 계속될 경우, 층간소음관리위원회에 조정을 신청할 수 있습니다.
   - **출처**: 공동주택관리법 제20조 (법령 조문번호: 제20조 ④, 제20조 ⑤)

3. **층간소음관리위원회의 조정 신청**: 관리주체나 윗집의 조치에도 불구하고 문제가 해결되지 않는다면, 공동주택 층간소음관리위원회의 조정을 신청할 수 있습니다.
   - **출처**: 공동주택관리법 제20조 (법령 조문번호: 제20조 ④)

4. **지방자치단체의 지원 활용**: 지방자치단체의 조례에 따라 층간소음 상담, 진단, 교육 등의 지원을 받을 수 있습니다. 이를 통해 윗집과의 분쟁 해결을 위한 중재 방안을 모색할 수 있습니다.
   - **출처**: 공동주택관리법 제34조의2 (법령 조문번호: 제34조의2)

5. **공식 분쟁 조정 기구 이용**: 층간소음 문제가 지속될 경우, 공동주택관리 분쟁조정위원회나 환경분쟁 조정법에 따른 환경분쟁조정위원회에 조정을 신청할 수 있습니다.
   - **출처**: 공동주택관리법 제40조의2 (법령 조문번호: 제40조의2) 및 환경분쟁 조정법 제4조 (법령 조문번호: 환경분쟁 조정법 제4조)

이러한 절차를 통해 윗집의 소음 문제를 해결하고자 노력할 수 있습니다.

---
**관련 법령/판례:**
- [법령] [공동주택 층간소음의 범위와 기준에 관한 규칙 제3조](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EA%B3%B5%EB%8F%99%EC%A3%BC%ED%83%9D%20%EC%B8%B5%EA%B0%84%EC%86%8C%EC%9D%8C%EC%9D%98%20%EB%B2%94%EC%9C%84%EC%99%80%20%EA%B8%B0%EC%A4%80%EC%97%90%20%EA%B4%80%ED%95%9C%20%EA%B7%9C%EC%B9%99/%EC%A0%9C3%EC%A1%B0)
- [법령] [공동주택관리법 제20조](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EA%B3%B5%EB%8F%99%EC%A3%BC%ED%83%9D%EA%B4%80%EB%A6%AC%EB%B2%95/%EC%A0%9C20%EC%A1%B0)

**검색 근거 목록** — 생성 모델에 최종 전달된 문맥 전체와는 구분합니다.

,title,doc_type,version,source_url
0,공동주택 층간소음의 범위와 기준에 관한 규칙,법령,공동주택 층간소음의 범위와 기준에 관한 규칙|제3조@2023-01-02,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EA%B...
1,공동주택관리법,법령,공동주택관리법|제20조@2026-07-01,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EA%B...
2,공동주택관리법,법령,공동주택관리법|제34조의2@2026-07-01,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EA%B...
3,공동주택관리법,법령,공동주택관리법|제85조@2026-07-01,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EA%B...
4,공동주택관리법,법령,공동주택관리법|제85조의2@2026-07-01,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EA%B...


원시 결과: Q07.json / usage: unknown / 비용: unknown


### 검토 메모 — 수정 필요

- **질문의 조건을 제대로 반영했는지:** 관리주체에 알리고 단계적으로 조정을 구하는 방향은 유용하다. 그러나 아파트인지 다가구 원룸인지도 확인하지 않고 모든 자취생에게 관리주체·위원회가 있는 것처럼 설명했다.
- **답변과 근거·인용이 일치하는지:** 공동주택관리법 제20조는 관련 근거다. 다만 ④·⑤항을 층간소음관리위원회 조정 근거라고 단정하고, 공식 분쟁조정 근거를 제40조의2로 표시한 것은 조문 대응 재검증이 필요하다. 제34조의2는 소규모 공동주택 지원이라는 적용 범위가 있다. 검색 목록의 2026-07-01 판본과 항 번호를 맞춰야 하며, 장래 시행 개정문을 현재 규정으로 섞지 않아야 한다.
- **필요한 추가 질문·예외가 있는지:** 주택 유형, 소음 종류(발걸음·음악·말소리·배관 등), 시간대·지속시간, 관리인 유무, 이미 요청한 조치를 묻는다. 소음 종류와 건물 유형에 따라 상담·측정 대상이 다르므로 기관을 일률적으로 연결하지 않는다.
- **설명이 쉽고 다음 행동을 알 수 있는지:** 소음 날짜·시간·지속시간과 집 안에서 들리는 상태를 기록 → 관리인에게 구체적으로 전달 → 대상에 해당하면 한국환경공단 층간소음 이웃사이센터에 상담 신청 → 지속 시 적합한 분쟁조정 창구 확인으로 단순화한다. 휴대전화 기록은 정황 자료이며 법정 기준 초과를 확정하는 전문 측정과 다르다.
- **심각한 오류·수정 의견:** ‘관리주체에게 알려야 한다’는 의무 표현을 ‘알리고 중재를 요청할 수 있다’로 고친다. 이웃사이센터는 공동주택이 아닌 원룸·단독주택·오피스텔 등이 업무 대상에서 제외될 수 있으므로 확인이 필요하다. 직접 대면 갈등이나 보복 소음을 부추기지 말고, 위협·폭력 등 긴급 상황은 소음 조정 절차와 구분한다.

점수: **3/5 — 공인중개사 자문을 반영한 확정 정성 점수**

**검토에 사용한 공식 자료** (2026-10-18 확인; 원래 답변의 검색 결과와 별도):

- [층간소음 상담·측정 절차](https://www.easylaw.go.kr/CSP/CnpClsMain.laf?ccfNo=2&cciNo=1&cnpClsNo=1&csmSeq=1890)
- [이웃사이센터 업무 대상·제외 유형](https://floor.noiseinfo.or.kr/floornoise/home/notice/faq.do?boardNo=140)
- [소규모 공동주택 지원 규정](https://www.law.go.kr/lsInfoP.do?lsiSeq=255407&viewCls=lsRvsDocInfoR)


## Q08

In [11]:
show_answer("Q08")

**질문**

공인중개사가 되려면 어떻게 준비해야돼?

,질문 ID,수집 상태,전체 시간(초),첫 토큰(초),전처리(초),검색 근거 수,인용 수,오류
7,Q08,ok,12.56,3.13,2.47,5,3,없음


**Before 답변 전문 — 모델 원문**

답변: 공인중개사가 되기 위한 준비 과정은 다음과 같습니다:

1. **자격 요건 충족**: 대한민국 국민이어야 하며, 특정 범죄 기록이 없어야 합니다. 또한, 부동산 관련 법규와 실무 지식을 갖추어야 합니다.

2. **교육 이수**: 공인중개사 자격시험에 응시하기 위해서는 관련 교육 과정을 이수해야 합니다. 이 교육은 일반적으로 공인중개사 시험을 주관하는 기관에서 제공하며, 부동산 관련 법규, 실무 지식, 윤리 등 다양한 주제를 다룹니다.

3. **자격시험 합격**: 한국공인중개사협회나 지정된 시험기관에서 실시하는 공인중개사 자격시험에 합격해야 합니다. 시험은 이론과 실무를 포함한 다양한 분야를 평가합니다.

4. **등록**: 시험에 합격한 후에는 해당 기관에 등록을 완료해야 합니다. 등록 절차를 통해 공인중개사 자격을 공식적으로 취득하게 됩니다.

**참고 법령**:
- 공인중개사법 제2조 (공인중개사의 정의)
- 공인중개사법 제10조 (자격시험)

이러한 과정을 통해 공인중개사로서의 자격을 갖추게 되며, 부동산 중개업무를 수행할 수 있습니다. 하지만 각 단계의 구체적인 요건과 절차는 시간이 지남에 따라 변경될 수 있으므로, 최신 정보는 관련 기관의 공식 웹사이트나 직접 문의를 통해 확인하는 것이 좋습니다.

---
**관련 법령/판례:**
- [법령] [민법 제488조](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%AF%BC%EB%B2%95/%EC%A0%9C488%EC%A1%B0)
- [판례] [대법원 2024다239364 손해배상(기)[공인중개사가 부동산 중개 과정에서 임대차보증금 반환채무가 매수인에게 면책적으로 인수되지 않는다는 사정을 설명할 주의의무가 인정되는지 여부가 문제된 사건] 2024다239364](https://www.law.go.kr/LSW/precInfoP.do?precSeq=241585)
- [법령] [부동산 거래신고 등에 관한 법률 제3조](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B6%80%EB%8F%99%EC%82%B0%20%EA%B1%B0%EB%9E%98%EC%8B%A0%EA%B3%A0%20%EB%93%B1%EC%97%90%20%EA%B4%80%ED%95%9C%20%EB%B2%95%EB%A5%A0/%EC%A0%9C3%EC%A1%B0)

**검색 근거 목록** — 생성 모델에 최종 전달된 문맥 전체와는 구분합니다.

,title,doc_type,version,source_url
0,민법,법령,민법|제488조@2026-03-17,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%A...
1,대법원 2024다239364 손해배상(기)[공인중개사가 부동산 중개 과정에서 임대차...,판례,판례|2024다239364,https://www.law.go.kr/LSW/precInfoP.do?precSeq...
2,부동산 거래신고 등에 관한 법률,법령,부동산 거래신고 등에 관한 법률|제3조@2017-01-20,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B...
3,민법,법령,민법|제93조@2026-03-17,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%A...
4,민법,법령,민법|제88조@2026-03-17,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%A...


원시 결과: Q08.json / usage: unknown / 비용: unknown


### 검토 메모 — 수정 필요 — 우선 수정

- **질문의 조건을 제대로 반영했는지:** 준비 절차를 나누었지만 응시 요건·교육·시험 시행기관·자격 취득과 개업 등록을 연속해서 잘못 안내했다. 이 답변대로라면 불필요한 교육을 먼저 찾거나 응시를 포기할 수 있다.
- **답변과 근거·인용이 일치하는지:** 공인중개사법 제10조는 중개사무소 등록 등의 결격사유에 관한 조문으로 ‘자격시험’ 조문이 아니다. 제4조와 Q-Net 시험 안내가 우선 근거다. 링크한 민법 제488조·중개상 설명의무 판례·거래신고법 제3조는 시험 준비를 뒷받침하지 않는다.
- **필요한 추가 질문·예외가 있는지:** 질문이 ‘시험 합격 준비’인지 ‘취업·개업 준비’인지 확인하되, 기본 시험 안내는 바로 제공한다. 일반적인 응시자격은 제한 없음이며 부정행위 제재·자격취소 후 제한·기자격 취득자 등 예외와 개설등록 결격사유를 구분한다.
- **설명이 쉽고 다음 행동을 알 수 있는지:** Q-Net 공인중개사에서 시행계획·접수 일정·기출문제를 확인하도록 연결한다. 1차는 부동산학개론과 민법·민사특별법, 2차는 중개사법령·중개실무, 공법, 공시법·세법 범위를 공부한다는 출발점을 제시한다. 시험 합격 후 취업인지 개업인지에 따라 다음 절차를 나눈다.
- **심각한 오류·수정 의견:** ‘대한민국 국민이어야 한다’, ‘응시 전 교육 필수’, ‘협회가 시험 실시’, ‘등록해야 자격 취득’은 수정한다. 시험은 한국산업인력공단 Q-Net 안내에 따라 준비하며, 실무교육과 관할 시·군·구의 사무소 개설등록은 개업 등 업무 단계의 별도 절차다. 범죄 이력의 응시·등록 영향도 뭉뚱그려 단정하지 않는다.

점수: **1/5 — 공인중개사 자문을 반영한 확정 정성 점수**

**검토에 사용한 공식 자료** (2026-10-18 확인; 원래 답변의 검색 결과와 별도):

- [Q-Net 공인중개사 응시자격·시험과목](https://www.q-net.or.kr/crf005.do?gId=08&gSite=L&id=crf00503s02&jmCd=9630&jmInfoDivCcd=B0&seriesCd=8)
- [실무교육: 공인중개사법 제34조](https://www.law.go.kr/LSW/lsLawLinkInfo.do?chrClsCd=010202&lsJoLnkSeq=900140549)
- [사무소 개설등록: 제9조](https://law.go.kr/lsLinkCommonInfo.do?chrClsCd=010202&lsJoLnkSeq=1024368625)


## Q09

In [12]:
show_answer("Q09")

**질문**

다른 사람이 집을 계약할까봐 보증금의 10프로는 먼저 지불했는데, 내가 계약을 못하게 되었을때 선지불했던 금액 다시 돌려받을 수 있을까?

,질문 ID,수집 상태,전체 시간(초),첫 토큰(초),전처리(초),검색 근거 수,인용 수,오류
8,Q09,ok,13.73,3.33,2.59,5,1,없음


**Before 답변 전문 — 모델 원문**

답변: 일반적으로 임차인이 계약을 성사시키지 못하고 계약 체결이 이루어지지 않은 경우, 선지불한 보증금의 일부인 10%에 대해서도 계약 조건에 따라 반환 여부가 결정됩니다. 주택임대차보호법에 명시된 내용만으로는 구체적인 반환 규정을 확인하기 어렵지만, 일반적인 관행과 계약 조건에 따라 다음과 같은 점을 고려해야 합니다:

1. **계약 조건 확인**: 계약서에 선지불 보증금의 반환 조건이 명시되어 있는지 확인해야 합니다. 많은 경우에 계약이 성사되지 않을 때 선지불 금액의 반환에 대한 조항이 포함되어 있습니다.

2. **법적 근거**: 주택임대차보호법 제3조의2는 주로 보증금의 회수와 우선변제권에 초점을 맞추고 있으며, 선지불 보증금의 반환에 대한 직접적인 규정은 명시되어 있지 않습니다. 따라서 계약 조건과 당사자 간 합의가 중요합니다.

3. **당사자 간 합의**: 계약 체결 전에 임대인과 임차인 간에 선지불 보증금의 반환 조건에 대한 합의가 이루어져 있다면, 그 합의에 따라 반환 여부가 결정됩니다.

결론적으로, 선지불한 금액의 반환 가능성은 주로 계약서에 명시된 조건과 당사자 간의 합의에 따라 달라집니다. 계약 조건을 다시 확인하고 필요하다면 임대인과 직접 협의하여 반환 여부를 결정하는 것이 좋습니다. 만약 불확실한 경우 법률 자문을 받는 것도 고려해볼 만합니다.

**출처:**
- 주택임대차보호법 (법령), 제3조의2 (보증금의 회수) 관련 조항들

---
**관련 법령/판례:**
- [법령] [주택임대차보호법 제3조의2](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A3%BC%ED%83%9D%EC%9E%84%EB%8C%80%EC%B0%A8%EB%B3%B4%ED%98%B8%EB%B2%95/%EC%A0%9C3%EC%A1%B0%EC%9D%982)

**검색 근거 목록** — 생성 모델에 최종 전달된 문맥 전체와는 구분합니다.

,title,doc_type,version,source_url
0,주택임대차보호법,법령,주택임대차보호법|제3조의2@2013-08-13,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...
1,주택임대차보호법,법령,주택임대차보호법|제3조의2@2015-07-01,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...
2,주택임대차보호법,법령,주택임대차보호법|제3조의2@2002-06-30,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...
3,주택임대차보호법,법령,주택임대차보호법|제3조의2@2016-12-01,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...
4,주택임대차보호법,법령,주택임대차보호법|제3조의2@2002-07-01,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...


원시 결과: Q09.json / usage: unknown / 비용: unknown


### 검토 메모 — 수정 필요

- **질문의 조건을 제대로 반영했는지:** 반환 여부가 합의 내용에 따라 달라진다는 신중함은 적절하다. 하지만 사용자가 말하지 않은 ‘계약 체결이 이루어지지 않았다’는 상태를 가정했다. 계약서 작성 전에도 집·보증금·월세·기간 등 핵심 조건의 합의로 계약이 성립할 수 있다.
- **답변과 근거·인용이 일치하는지:** 주택임대차보호법 제3조의2가 선지급 계약금 반환의 직접 규정이 아니라는 설명은 맞다. 그러나 그 조문만 인용하고 끝내면 핵심 법리가 비어 있다. 민법 제565조와 제567조에 따른 계약금 해제, 계약 성립 여부와 금원의 성격을 검토해야 한다. ‘많은 경우 반환 조항이 포함된다’는 관행 주장도 근거 없이 넣지 않는다.
- **필요한 추가 질문·예외가 있는지:** 누구에게 얼마를 보냈는지, 어떤 집·금액·기간에 합의했는지, 문자·메신저에 ‘가계약금/계약금/환불’ 약정이 있는지, 계약금 전액인지 일부인지, 중도금 등 이행 착수 여부, 계약을 못 하는 이유가 무엇인지 확인한다.
- **설명이 쉽고 다음 행동을 알 수 있는지:** 송금내역과 송금 전후 문자·중개사 설명을 보관하고 ‘무슨 합의에 따라 어느 금액을 반환해 달라는지’를 정리하도록 안내한다. 보증금의 10%라는 비율만으로 전액 환급 또는 몰수를 확정하지 않는다.
- **심각한 오류·수정 의견:** 조건별 결론을 보여줘야 한다. 계약이 성립하고 지급금이 해약금 성격의 계약금이면 다른 약정이 없는 한 상대방 등의 이행 착수 전 계약금 포기로 해제하는 문제가 생길 수 있다. 계약 미성립·환급 약정·상대방 채무불이행 등은 별도 검토한다. ‘가계약’이라는 이름이나 계약서 미작성만으로 환급된다고 안내해서는 안 된다.

점수: **2/5 — 공인중개사 자문을 반영한 확정 정성 점수**

**보류 사항:** 사실관계·계약 연도·적용 판본 등 미확정 조건을 유지합니다. 필요한 추가 질문과 조건부 설명은 검토할 수 있으나 특정 조건의 검색 점수는 확정 전 보류합니다.

**검토에 사용한 공식 자료** (2026-10-18 확인; 원래 답변의 검색 결과와 별도):

- [민법 제565조: 해약금](https://www.law.go.kr/LSW/lsInfoP.do?docType=JO&joNo=010300&lsiSeq=188377)
- [민법 제567조: 유상계약에의 준용](https://www.easylaw.go.kr/CSP/CnpClsMain.laf?ccfNo=4&cciNo=1&cnpClsNo=1&csmSeq=1171&popMenu=ov)


## Q10

In [13]:
show_answer("Q10")

**질문**

혹시 자취생을 위한 주거비 지원 정책 좀 아는 거 있니?

,질문 ID,수집 상태,전체 시간(초),첫 토큰(초),전처리(초),검색 근거 수,인용 수,오류
9,Q10,ok,13.96,3.38,2.31,5,4,없음


**Before 답변 전문 — 모델 원문**

답변: 제공된 참고 문서들은 주로 공동주택 관리, 임대차 계약 관련 법적 조항, 그리고 난민 지위에 있는 외국인의 공공임대주택 입주 자격에 대한 판례 내용을 다루고 있습니다. 자취생을 위한 직접적인 주거비 지원 정책에 대한 정보는 포함되어 있지 않습니다.

그러나 일반적으로 자취생을 위한 주거비 지원 정책은 다음과 같은 형태로 존재할 수 있습니다:

1. **공공임대주택**: 정부나 지자체가 운영하는 공공임대주택 프로그램은 저렴한 임대료로 주택을 제공하며, 소득 기준에 따라 지원 대상이 될 수 있습니다. (참조: 공공주택 특별법, 주택임대차보호법 관련 조항)

2. **학자금 지원**: 일부 지자체나 정부 프로그램에서는 자취생이나 대학생을 위한 주거비 지원을 포함한 학자금 지원을 제공할 수 있습니다. 이는 소득 수준, 학업 성적, 가구 상황 등을 고려하여 결정됩니다.

3. **청년 주거 지원**: 청년을 대상으로 하는 주거 지원 정책이 있을 수 있습니다. 예를 들어, 청년 주거안정자금이나 청년 전세대출 등이 있으며, 이는 청년의 경제적 부담을 줄이기 위해 설계된 프로그램입니다. (참고로, 이 내용은 제공된 문서에는 포함되어 있지 않으나 일반적인 정책 범위 내에서 언급됩니다.)

구체적인 정책 내용과 지원 자격은 해당 지역의 지자체나 정부 웹사이트를 통해 확인하거나, 관련 부서에 문의하는 것이 가장 정확한 정보를 얻는 방법입니다. 제공된 문서들만으로는 자취생을 위한 특정 주거비 지원 정책에 대한 자세한 내용을 안내하기 어렵습니다.

---
**관련 법령/판례:**
- [법령] [공동주택관리법 제21조](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EA%B3%B5%EB%8F%99%EC%A3%BC%ED%83%9D%EA%B4%80%EB%A6%AC%EB%B2%95/%EC%A0%9C21%EC%A1%B0)
- [판례] [서울서부지방법원 2016가단232170 임대차보증금 2016가단232170](https://www.law.go.kr/LSW/precInfoP.do?precSeq=224423)
- [법령] [주택임대차보호법 제1조](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A3%BC%ED%83%9D%EC%9E%84%EB%8C%80%EC%B0%A8%EB%B3%B4%ED%98%B8%EB%B2%95/%EC%A0%9C1%EC%A1%B0)
- [판례] [서울행법 2020구합78100 전세임대주택신청거부처분취소청구의소 2020구합78100](https://www.law.go.kr/LSW/precInfoP.do?precSeq=219903)

**검색 근거 목록** — 생성 모델에 최종 전달된 문맥 전체와는 구분합니다.

,title,doc_type,version,source_url
0,공동주택관리법,법령,공동주택관리법|제21조@2026-07-01,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EA%B...
1,서울서부지방법원 2016가단232170 임대차보증금,판례,판례|2016가단232170,https://www.law.go.kr/LSW/precInfoP.do?precSeq...
2,주택임대차보호법,법령,주택임대차보호법|제1조@2010-11-18,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...
3,주택임대차보호법,법령,주택임대차보호법|제1조@2008-03-21,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...
4,서울행법 2020구합78100 전세임대주택신청거부처분취소청구의소,판례,판례|2020구합78100,https://www.law.go.kr/LSW/precInfoP.do?precSeq...


원시 결과: Q10.json / usage: unknown / 비용: unknown


### 검토 메모 — 수정 필요

- **질문의 조건을 제대로 반영했는지:** 검색된 자료에 직접적인 주거비 지원 정보가 없다고 인정한 점은 좋다. 다만 지원 제도가 ‘있을 수 있다’는 추상적 나열에 머물러 사용자가 받을 수 있는 지원을 좁혀주지 못했다.
- **답변과 근거·인용이 일치하는지:** 공동주택관리법·임대차보증금 판례·주택임대차보호법 목적 조문은 월세 지원이나 청년대출의 모집 조건을 입증하지 못한다. 외부 자료가 필요하다고 설명하면서 무관한 출처를 정책 근거처럼 붙이면 안 된다. 원래 답변 생성 당시에는 외부 검색 미연결이라는 제한을 유지한다.
- **필요한 추가 질문·예외가 있는지:** 거주 지역, 나이, 학생/취업 여부, 무주택 여부, 보증금·월세, 본인 및 가구 소득·자산, 부모와 별도 거주 여부를 단계적으로 묻는다. 우선 지역·나이·보증금/월세만 받아 후보를 줄이고 나머지는 해당 공고의 필수 요건일 때 확인한다.
- **설명이 쉽고 다음 행동을 알 수 있는지:** ‘지원금’과 ‘갚아야 하는 대출’, ‘공공임대 입주’를 나누고 마이홈 주거지원 자가진단과 거주 지자체 공고로 연결한다. 상담원이 확인할 항목은 모집 중인지·소득/자산 기준·중복 수혜 제한·신청 기한·준비 서류다. ‘정부 사이트 확인’만 말하는 것보다 실제 다음 행동이 분명하다.
- **심각한 오류·수정 의견:** ‘학자금 지원에 주거비가 포함될 수 있다’는 막연한 설명과 확인되지 않은 사업 명칭은 빼거나 확인 전 후보로 표시한다. 이번 검토에서 마이홈 안내 경로는 확인했지만 개별 공고의 모집 상태·지원액·자격을 확인한 것은 아니다. 현재 신청 가능하다는 결론이나 그에 대한 점수를 새로 부여하지 않는다.

점수: **2/5 — 공인중개사 자문을 반영한 확정 정성 점수**

**보류 사항:** 외부 검색 도구는 연결되지 않았습니다. 공고 확인이 필요한 사실의 정확성 점수는 보류하고, 추가 질문과 근거 없는 단정 여부를 검토합니다.

추가 보류: 이번 자문에서는 공식 안내 경로만 확인했습니다. 개별 모집 공고의 현재 신청 가능 여부·지원액·자격 정확성은 계속 보류합니다. 위 점수는 한계 고지·질문 구체화·행동 안내를 평가한 것으로 정책 사실 정확성의 확정점수가 아닙니다.

**검토에 사용한 공식 자료** (2026-10-18 확인; 원래 답변의 검색 결과와 별도):

- [마이홈 주거복지 안내](https://www.myhome.go.kr/hws/portal/main/selectMyHomeInfoView.do)
- [청년·신혼부부 주거지원 자가진단](https://m.myhome.go.kr/hws/mbl/dgn/selectSelfDiagnosisYgJindanView.do)


## Q11

In [14]:
show_answer("Q11")

**질문**

계약이 만료되어가는 시점이고 나는 계약갱신은 안할건데, 집주인이 사전에 집보러 오는 사람과 언제 올거다라는 얘기도 없이 마음대로 아직 내가 살고 있는 집을 들락날락 할 수 있는거야?

,질문 ID,수집 상태,전체 시간(초),첫 토큰(초),전처리(초),검색 근거 수,인용 수,오류
10,Q11,ok,9.23,3.42,2.83,5,3,없음


**Before 답변 전문 — 모델 원문**

답변: 일반적으로 계약이 만료되는 시점에 도달했을 때, 임차인은 계약갱신 요구권이 없다면 임대인의 동의 없이 임차인의 거주권을 보장받습니다. 그러나 임대인이 집을 보러 오거나 관리 목적으로 출입하고자 할 경우, 민법 제629조에 따르면 임대인의 동의 없이 임차인은 임차물을 전대하거나 임대인의 출입을 제한할 수 없습니다. 따라서 임대인이 사전에 통보하지 않았더라도, 임대인의 합법적인 출입은 허용됩니다. 하지만 임대인의 출입이 빈번하거나 불합리하게 이루어질 경우, 임차인은 이에 대해 불편함을 느끼거나 법적 조언을 구하는 것이 좋습니다. 중요한 것은 임대인이 계약 조건을 위반하지 않는 한, 임차인의 거주 공간을 존중받을 권리가 있다는 점입니다.

**출처:**
- 민법 제629조 (임차권의 양도, 전대의 제한)

---
**관련 법령/판례:**
- [법령] [부동산 거래신고 등에 관한 법률 시행령 제14조의2](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B6%80%EB%8F%99%EC%82%B0%20%EA%B1%B0%EB%9E%98%EC%8B%A0%EA%B3%A0%20%EB%93%B1%EC%97%90%20%EA%B4%80%ED%95%9C%20%EB%B2%95%EB%A5%A0%20%EC%8B%9C%ED%96%89%EB%A0%B9/%EC%A0%9C14%EC%A1%B0%EC%9D%982)
- [법령] [민법 제629조](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%AF%BC%EB%B2%95/%EC%A0%9C629%EC%A1%B0)
- [법령] [주택임대차보호법 제6조의3](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A3%BC%ED%83%9D%EC%9E%84%EB%8C%80%EC%B0%A8%EB%B3%B4%ED%98%B8%EB%B2%95/%EC%A0%9C6%EC%A1%B0%EC%9D%983)

**검색 근거 목록** — 생성 모델에 최종 전달된 문맥 전체와는 구분합니다.

,title,doc_type,version,source_url
0,부동산 거래신고 등에 관한 법률 시행령,법령,부동산 거래신고 등에 관한 법률 시행령|제14조의2@2026-10-01,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B...
1,민법,법령,민법|제629조@2026-03-17,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%A...
2,주택임대차보호법,법령,주택임대차보호법|제6조의3@2020-07-31,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...
3,민법,법령,민법|제639조@2026-03-17,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%A...
4,부동산 거래신고 등에 관한 법률 시행령,법령,부동산 거래신고 등에 관한 법률 시행령|제14조의2@2026-05-29,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B...


원시 결과: Q11.json / usage: unknown / 비용: unknown


### 검토 메모 — 수정 필요 — 우선 수정

- **질문의 조건을 제대로 반영했는지:** ‘아직 거주 중인데 집주인이 예고·동의 없이 다른 사람과 들어오는 상황’을 다루어야 한다. 답변은 계약갱신요구권 여부를 끌어오고 오히려 임차인이 출입을 제한할 수 없다고 잘못 안내했다.
- **답변과 근거·인용이 일치하는지:** 민법 제629조는 임차권 양도·전대 제한 규정이다. 임대인의 무단 출입 권한을 부여하는 조문이 아니다. 거래신고 시행령과 갱신요구권 조문도 그 결론을 뒷받침하지 않는다. 단순한 근거 부족을 넘어 조문에 없는 내용을 만들어 핵심 결론을 뒤집었다.
- **필요한 추가 질문·예외가 있는지:** 실제 출입인지 방문 요구인지, 사전 동의 또는 집 보여주기 특약이 있는지, 비밀번호를 이용해 부재 중 들어왔는지, 누수·화재 같은 긴급 사유인지 확인한다. 주택 보존에 필요한 행위의 수인 문제와 새 임차인에게 집을 보여주는 일을 구별한다.
- **설명이 쉽고 다음 행동을 알 수 있는지:** ‘집주인이라도 거주 중인 집에 마음대로 들어올 수 있는 것은 아닙니다. 방문 전 날짜·시간·방문자를 정하고 동의를 받도록 문자로 요청하세요’라고 먼저 답해야 한다. 출입 일시·문자·가능한 기록을 보관하고, 방문 가능한 시간대를 제안해 협의하도록 한다. 침입이 진행 중이거나 위협을 받으면 경찰 신고로 연결한다.
- **심각한 오류·수정 의견:** ‘사전 통보가 없어도 허용’ 문장은 즉시 삭제한다. 소유자라도 출입 경위와 방식에 따라 형법 제319조 주거침입 문제가 생길 수 있다. 사전 통보만으로 동의가 대체되는 것도 아니다. 긴급 보존행위 등 예외는 별도 판단하며, 모든 방문을 자동으로 범죄라고 단정하지 않는다.

점수: **1/5 — 공인중개사 자문을 반영한 확정 정성 점수**

**검토에 사용한 공식 자료** (2026-10-18 확인; 원래 답변의 검색 결과와 별도):

- [민법 제629조의 실제 내용](https://www.law.go.kr/LSW/lsSideInfoP.do?docCls=jo&joBrNo=00&joNo=0629&lsiSeq=284415&urlMode=lsScJoRltInfoR)
- [형법 제319조: 주거침입·퇴거불응](https://www.law.go.kr/lsLinkCommonInfo.do?chrClsCd=010202&lsJoLnkSeq=1031456705)


## Q12

In [15]:
show_answer("Q12")

**질문**

월세 계약서에 입주일자는 5월1일로 한다고 쓰고 계약서를 쓴 날짜가 3월25일 이라면, 임대차 계약 신고는 3월 25일부터 한달이내에 해야 하는건가요 아니면 입주일자인 5월 4일부터 해야하는건가요?

,질문 ID,수집 상태,전체 시간(초),첫 토큰(초),전처리(초),검색 근거 수,인용 수,오류
11,Q12,ok,5.9,2.68,2.43,5,3,없음


**Before 답변 전문 — 모델 원문**

답변: 임대차 계약 신고는 **계약서 체결일**부터 한 달 이내에 이루어져야 합니다. 따라서, 월세 계약서를 작성한 날짜가 3월 25일이므로, 이 경우 임대차 계약 신고는 **3월 25일부터 4월 24일 이내**에 해야 합니다. 입주일자인 5월 1일은 신고 기한과 직접적인 관련이 없습니다.

**출처:**
- 부동산 거래신고 등에 관한 법률 제6조의2 (법령) ① 항목

---
**관련 법령/판례:**
- [법령] [부동산 거래신고 등에 관한 법률 제6조의2](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B6%80%EB%8F%99%EC%82%B0%20%EA%B1%B0%EB%9E%98%EC%8B%A0%EA%B3%A0%20%EB%93%B1%EC%97%90%20%EA%B4%80%ED%95%9C%20%EB%B2%95%EB%A5%A0/%EC%A0%9C6%EC%A1%B0%EC%9D%982)
- [법령] [주택임대차보호법 제3조의7](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A3%BC%ED%83%9D%EC%9E%84%EB%8C%80%EC%B0%A8%EB%B3%B4%ED%98%B8%EB%B2%95/%EC%A0%9C3%EC%A1%B0%EC%9D%987)
- [법령] [민법 제635조](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%AF%BC%EB%B2%95/%EC%A0%9C635%EC%A1%B0)

**검색 근거 목록** — 생성 모델에 최종 전달된 문맥 전체와는 구분합니다.

,title,doc_type,version,source_url
0,부동산 거래신고 등에 관한 법률,법령,부동산 거래신고 등에 관한 법률|제6조의2@2021-06-01,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B...
1,부동산 거래신고 등에 관한 법률,법령,부동산 거래신고 등에 관한 법률|제6조의3@2021-06-01,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B...
2,부동산 거래신고 등에 관한 법률,법령,부동산 거래신고 등에 관한 법률|제6조의5@2021-06-01,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B...
3,주택임대차보호법,법령,주택임대차보호법|제3조의7@2023-04-18,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...
4,민법,법령,민법|제635조@2026-03-17,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%A...


원시 결과: Q12.json / usage: unknown / 비용: unknown


### 검토 메모 — 수정 필요

- **질문의 조건을 제대로 반영했는지:** 신고 대상 계약이라면 입주일이 아닌 계약 체결일을 기준으로 한다는 핵심은 맞다. 다만 계약 연도·소재지·금액을 확인하지 않고 신고 의무와 마감일을 확정했다. 질문 안의 5월 1일/5월 4일 불일치도 임의로 하나로 정리하지 말아야 한다.
- **답변과 근거·인용이 일치하는지:** 부동산거래신고법 제6조의2는 계약 체결일부터 30일 이내라는 설명에 직접 부합한다. 주택임대차보호법 제3조의7과 민법 제635조는 그 기한 산정의 직접 근거가 아니다. 계약 연도를 특정하지 않고 2021-06-01 판본만으로 의무를 확정할 수 없다.
- **필요한 추가 질문·예외가 있는지:** 계약 연도, 실제 합의·체결일이 3월 25일인지, 지역, 보증금·월세, 신규/변경 없는 갱신인지 확인한다. 일반 주택 임대차 신고제는 2021년 6월 1일 이후 체결 계약부터 검토하며, 해당 지역·금액 기준과 적용 판본을 함께 확인한다.
- **설명이 쉽고 다음 행동을 알 수 있는지:** ‘신고 대상이면 3월 25일 계약 체결일부터 30일 이내입니다. 입주가 5월이어도 기다리지 마세요’로 시작하고 계약서 준비 → 관할 주민센터 또는 부동산거래관리시스템에서 신고 → 접수·처리 여부 확인까지 안내한다. 임대차 신고와 입주 후 전입신고는 서로 다른 절차라고 알려준다.
- **심각한 오류·수정 의견:** ‘한 달 이내’는 ‘30일 이내’로 고친다. 초일 불산입을 전제로 3월 25일 다음 날부터 30일째는 4월 24일이지만, 말일이 토요일·공휴일인지에 따라 달라질 수 있어 연도 없이 확정하면 안 된다. ‘4월 24일’ 자체를 일수 계산 오류로 판정하는 것도 부적절하다. 질문의 두 입주일은 어느 쪽이든 신고 기산점에는 영향을 주지 않는다.

점수: **3/5 — 공인중개사 자문을 반영한 확정 정성 점수**

**보류 사항:** 사실관계·계약 연도·적용 판본 등 미확정 조건을 유지합니다. 필요한 추가 질문과 조건부 설명은 검토할 수 있으나 특정 조건의 검색 점수는 확정 전 보류합니다.

**검토에 사용한 공식 자료** (2026-10-18 확인; 원래 답변의 검색 결과와 별도):

- [신고 시행일·대상·방법](https://easylaw.go.kr/CSP/CnpClsMain.laf?ccfNo=2&cciNo=2&cnpClsNo=5&csmSeq=629&menuType=cnpcls)
- [민법 제157조·제161조: 초일과 말일 처리](https://www.law.go.kr/LSW/lsInfoP.do?docType=JO&joNo=010300&lsiSeq=188377)


## T01

In [16]:
show_answer("T01")

**질문**

2019년 3월에 2년 전세 계약을 했고 2021년 3월에 만료됐어. 만료 몇 달 전에 계약갱신요구권을 쓸 수 있었던 거야?

,질문 ID,수집 상태,전체 시간(초),첫 토큰(초),전처리(초),검색 근거 수,인용 수,오류
12,T01,ok,5.72,3.03,2.59,5,3,없음


**Before 답변 전문 — 모델 원문**

답변: 임차인은 계약갱신요구권을 **임차 기간 만료 1개월 전부터** 행사할 수 있습니다. 따라서 2021년 3월에 만료되는 계약의 경우, **2021년 2월부터** 계약갱신요구권을 행사할 수 있었습니다.

**출처:** 주택임대차보호법 제6조의3 제1항 (법령)

---
**관련 법령/판례:**
- [법령] [주택임대차보호법 제6조의3](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A3%BC%ED%83%9D%EC%9E%84%EB%8C%80%EC%B0%A8%EB%B3%B4%ED%98%B8%EB%B2%95/%EC%A0%9C6%EC%A1%B0%EC%9D%983)
- [법령] [민법 제620조](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%AF%BC%EB%B2%95/%EC%A0%9C620%EC%A1%B0)
- [법령] [부동산 거래신고 등에 관한 법률 시행령 제14조의2](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B6%80%EB%8F%99%EC%82%B0%20%EA%B1%B0%EB%9E%98%EC%8B%A0%EA%B3%A0%20%EB%93%B1%EC%97%90%20%EA%B4%80%ED%95%9C%20%EB%B2%95%EB%A5%A0%20%EC%8B%9C%ED%96%89%EB%A0%B9/%EC%A0%9C14%EC%A1%B0%EC%9D%982)

**검색 근거 목록** — 생성 모델에 최종 전달된 문맥 전체와는 구분합니다.

,title,doc_type,version,source_url
0,주택임대차보호법,법령,주택임대차보호법|제6조의3@2020-07-31,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...
1,민법,법령,민법|제620조@2026-03-17,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%A...
2,부동산 거래신고 등에 관한 법률 시행령,법령,부동산 거래신고 등에 관한 법률 시행령|제14조의2@2026-02-27,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B...
3,부동산 거래신고 등에 관한 법률 시행령,법령,부동산 거래신고 등에 관한 법률 시행령|제14조의2@2026-05-29,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B...
4,민법,법령,민법|제312조@2026-03-17,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%A...


원시 결과: T01.json / usage: unknown / 비용: unknown


### 검토 메모 — 수정 필요 — 우선 수정

- **질문의 조건을 제대로 반영했는지:** 2019년 3월 체결·2021년 3월 만료라는 명확한 시간 조건을 놓쳤다. ‘만료 1개월 전부터’라고 하여 행사 마감 시점을 시작 시점으로 뒤집었다.
- **답변과 근거·인용이 일치하는지:** 주택임대차보호법 제6조의3만 연결하면 부족하다. 제6조의 행사 기간과 2020년 개정 부칙의 적용례를 함께 보아야 한다. 검색 목록에는 2020-07-31 판본이 있으나 답변은 그 기간을 제대로 적용하지 못했다. 민법 제620조와 거래신고 시행령은 행사 기간의 직접 근거가 아니다.
- **필요한 추가 질문·예외가 있는지:** 정확한 만료일과 중간에 새로 체결·갱신한 적이 있는지만 확인하면 일 단위 기한을 계산할 수 있다. 질문에 제시된 최초 계약이 그대로 존속했다는 전제에서 설명하며, 이미 주어진 연도 자체를 불명확하다고 보류하지 않는다.
- **설명이 쉽고 다음 행동을 알 수 있는지:** ‘주어진 계약이 그대로 유지됐다면 만료 6개월 전부터 1개월 전까지 요구할 수 있었어요. 2021년 3월 만료라면 대략 2020년 9월~2021년 2월 구간이고, 정확한 날짜는 계약 만료일을 기준으로 계산해요’라고 고친다. 의사표시가 임대인에게 도달한 기록을 남기는 방법도 안내한다.
- **심각한 오류·수정 의견:** 2020년 12월 10일 이후 최초 체결·갱신 계약부터 적용되는 ‘2개월 전’ 규칙을 2019년 계약에 소급 적용해서도 안 된다. 갱신요구권 제도는 2020년 7월 31일 당시 존속하던 임대차에도 적용된다는 점과 통지 기간 개정의 적용례를 분리해 설명해야 한다.

점수: **1/5 — 공인중개사 자문을 반영한 확정 정성 점수**

**검토에 사용한 공식 자료** (2026-10-18 확인; 원래 답변의 검색 결과와 별도):

- [정부 설명: 6개월~1개월 및 2020-12-10 이후 계약 적용례](https://www.korea.kr/news/policyNewsView.do?newsId=148875320)
- [생활법령: 기존 계약 적용과 기간 개정 구분](https://www.easylaw.go.kr/CSP/CnpClsMainBtr.laf?ccfNo=4&cciNo=4&cnpClsNo=1&csmSeq=629)


## T02

In [17]:
show_answer("T02")

**질문**

2021년 5월에 월세 계약을 했는데, 그때도 임대차 계약 신고를 해야 했어?

,질문 ID,수집 상태,전체 시간(초),첫 토큰(초),전처리(초),검색 근거 수,인용 수,오류
13,T02,ok,10.95,4.58,2.67,5,5,없음


**Before 답변 전문 — 모델 원문**

답변: 2021년 5월에 월세 계약을 체결한 경우, 해당 계약이 대통령령으로 정하는 금액을 초과하는 경우에만 주택 임대차 계약 신고 의무가 적용됩니다. 「부동산 거래신고 등에 관한 법률」 제6조의2에 따르면, 임대차 계약의 신고는 체결된 계약의 보증금 또는 차임이 대통령령으로 정하는 금액을 초과할 때 이루어져야 합니다. 

구체적인 금액 기준은 대통령령에 따라 정해지며, 해당 기준을 초과하지 않는다면 임대차 계약 신고를 하지 않아도 됩니다. 따라서, 신고 의무가 발생하는지 여부는 계약된 월세 금액과 대통령령에서 정한 금액 기준을 비교하여 판단해야 합니다. 만약 금액 기준을 초과한다면, 계약 체결일로부터 30일 이내에 주택 소재지를 관할하는 신고관청에 신고해야 합니다. 

정확한 금액 기준은 관련 대통령령을 확인해야 합니다. 만약 금액 기준을 초과한다면, 신고는 필수적입니다.

---
**관련 법령/판례:**
- [법령해석례] [법령해석례 22-0680 민원인 - 100세대 이상의 공동주택을 임대하는 임대사업자가 종전 임차인과의 임대차계약이 만료되어 새로운 임차인과 임대차계약을 체결하는 경우 임대차계약의 신고 기한(「민간임대주택에 관한 특별법」 제46조 등 관련) 22-0680](https://www.law.go.kr/LSW/expcInfoP.do?expcSeq=335271)
- [법령해석례] [법령해석례 20-0049 민원인 - 임대사업자 등록 전에 체결한 임대차계약의 임대료를 임대사업자 등록 후 신고한 경우 신고한 임대료가 최초 임대료에 해당하는지 여부(법률 제16000호 「민간임대주택에 관한 특별법」 제44조 등 관련) 20-0049](https://www.law.go.kr/LSW/expcInfoP.do?expcSeq=331229)
- [법령] [부동산 거래신고 등에 관한 법률 제6조의2](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B6%80%EB%8F%99%EC%82%B0%20%EA%B1%B0%EB%9E%98%EC%8B%A0%EA%B3%A0%20%EB%93%B1%EC%97%90%20%EA%B4%80%ED%95%9C%20%EB%B2%95%EB%A5%A0/%EC%A0%9C6%EC%A1%B0%EC%9D%982)
- [법령] [주택임대차보호법 제3조의7](https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A3%BC%ED%83%9D%EC%9E%84%EB%8C%80%EC%B0%A8%EB%B3%B4%ED%98%B8%EB%B2%95/%EC%A0%9C3%EC%A1%B0%EC%9D%987)
- [판례] [서울고등법원 2022나2043727 임대차보증금등반환청구의소 2022나2043727](https://www.law.go.kr/LSW/precInfoP.do?precSeq=238469)

**검색 근거 목록** — 생성 모델에 최종 전달된 문맥 전체와는 구분합니다.

,title,doc_type,version,source_url
0,법령해석례 22-0680 민원인 - 100세대 이상의 공동주택을 임대하는 임대사업자...,법령해석례,해석례|22-0680,https://www.law.go.kr/LSW/expcInfoP.do?expcSeq...
1,법령해석례 20-0049 민원인 - 임대사업자 등록 전에 체결한 임대차계약의 임대료...,법령해석례,해석례|20-0049,https://www.law.go.kr/LSW/expcInfoP.do?expcSeq...
2,부동산 거래신고 등에 관한 법률,법령,부동산 거래신고 등에 관한 법률|제6조의2@2021-06-01,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EB%B...
3,주택임대차보호법,법령,주택임대차보호법|제3조의7@2023-04-18,https://www.law.go.kr/%EB%B2%95%EB%A0%B9/%EC%A...
4,서울고등법원 2022나2043727 임대차보증금등반환청구의소,판례,판례|2022나2043727,https://www.law.go.kr/LSW/precInfoP.do?precSeq...


원시 결과: T02.json / usage: unknown / 비용: unknown


### 검토 메모 — 수정 필요 — 우선 수정

- **질문의 조건을 제대로 반영했는지:** ‘2021년 5월에 체결’이라는 핵심 조건을 무시하고 금액 기준만 안내했다. 일반 주택 임대차 신고제 시행 전 계약인지가 먼저 판단할 사항이다.
- **답변과 근거·인용이 일치하는지:** 인용 목록의 제6조의2@2021-06-01만 보더라도 계약 시점과 적용 여부를 확인해야 한다. 부동산거래신고법 개정 부칙과 국토교통부 시행 안내가 직접 근거다. 민간임대주택법상 등록 임대사업자의 별도 신고 해석례를 일반 임차인의 신고 의무와 섞었다.
- **필요한 추가 질문·예외가 있는지:** 일반적인 주택 임대차 계약 신고제를 묻는 것인지, 등록 임대사업자의 별도 신고를 묻는 것인지 필요할 때 구분한다. 2021년 6월 1일 이후 변경·갱신 계약이 있었다면 그 계약은 별도로 판단한다. 최초 5월 계약에 대한 기본 결론은 금액 정보 없이도 제시할 수 있다.
- **설명이 쉽고 다음 행동을 알 수 있는지:** ‘일반 전월세 신고제라면 2021년 6월 1일부터 체결된 계약이 대상이므로, 2021년 5월에 체결한 그 계약은 대상이 아니었어요. 입주일이 6월 이후라는 이유만으로 달라지지는 않아요’라고 먼저 답한다. 이후 변경·갱신이 있었는지만 후속 확인하면 된다.
- **심각한 오류·수정 의견:** ‘기준 금액을 초과하면 2021년 5월 계약도 신고 필수’라는 결론은 삭제한다. 시행 전 비대상과 시행 후 신고 의무는 있으나 과태료만 유예된 계도기간을 구분해야 한다. 과거 비대상 계약에 오늘의 신고 의무를 소급해 안내해서는 안 된다.

점수: **1/5 — 공인중개사 자문을 반영한 확정 정성 점수**

**검토에 사용한 공식 자료** (2026-10-18 확인; 원래 답변의 검색 결과와 별도):

- [국토교통부: 2021-06-01 신고제 시행](https://www.molit.go.kr/USR/NEWS/m_71/dtl.jsp?id=95085588)
- [시행 후 체결 계약 적용·부칙 안내](https://easylaw.go.kr/CSP/CnpClsMain.laf?ccfNo=2&cciNo=2&cnpClsNo=5&csmSeq=629&menuType=cnpcls)


## 검증 기록
원시 결과를 읽어 14개 ID·질문 일치와 재구성 Before 표시를 확인합니다. 모델 호출이나 자동 채점은 하지 않습니다. 검토 메모는 코드 실행으로 수정하지 않습니다. 전체 원시 답변·문맥과 실행 파일은 Git 밖의 백업에 보존합니다.